In [ ]:
#pip install datasets

In [1]:
import pandas as pd
from datasets import load_dataset

### Loading ade_corpus_v2_drug_ade_relation dataset

In [4]:
from datasets import load_dataset
ds_drug_ade = load_dataset("ade-benchmark-corpus/ade_corpus_v2", "Ade_corpus_v2_drug_ade_relation")

In [6]:
print(ds_drug_ade)

DatasetDict({
    train: Dataset({
        features: ['text', 'drug', 'effect', 'indexes'],
        num_rows: 6821
    })
})


In [8]:
train_df_drug_ade = pd.DataFrame(ds_drug_ade['train'][:])
train_df_drug_ade = train_df_drug_ade[['drug','effect']]
train_df_drug_ade.head()

,drug,effect
0,azithromycin,ototoxicity
1,dihydrotachysterol,increased calcium-release
2,dihydrotachysterol,hypercalcemia
3,naproxen,pseudoporphyria
4,oxaprozin,pseudoporphyria


In [10]:
# Check missing values per column
print(train_df_drug_ade.isnull().sum())

drug      0
effect    0
dtype: int64


In [12]:
print(train_df_drug_ade['effect'].unique())

['ototoxicity' 'increased calcium-release' 'hypercalcemia' ...
 'diabetes insipidus-like syndrome' 'lithium intoxication' 'Eosinophilia']


In [14]:
num_effect_ade = train_df_drug_ade['effect'].nunique()
print(num_effect_ade)

3341


In [16]:
train_df_drug_ade.shape

(6821, 2)

### Loading CADEC.v2 dataset

In [19]:
import os
import pandas as pd
import re

folder_path = '/Users/farhanaalam/Downloads/NLP/NLP_Project/CADEC.v2/sct' 

rows = []

for filename in os.listdir(folder_path):
    if filename.endswith('.ann'):
        file_path = os.path.join(folder_path, filename)
        with open(file_path, 'r', encoding='utf-8', errors='ignore') as f:
            lines = f.readlines()
            for line in lines:
                line = line.strip()
                if not line:
                    continue  # skip empty lines

                try:
                    parts = [p.strip() for p in line.split('|')]

                    if len(parts) >= 3:
                        id_and_concept = parts[0].split()
                        annotation_id = id_and_concept[0]
                        concept_id = id_and_concept[1] if len(id_and_concept) > 1 else None
                        term = parts[1]
                        
                        offset_text = parts[2]
                        
                        # Extract offsets and text separately
                        match = re.match(r'([\d\s;]+)\s+(.*)', offset_text)
                        if match:
                            offset_part = match.group(1).strip()
                            text_part = match.group(2).strip()
                        else:
                            offset_part = offset_text
                            text_part = ''

                        # Now deal with multiple spans
                        spans = []
                        for span in offset_part.split(';'):
                            span = span.strip()
                            if span:
                                nums = span.split()
                                if len(nums) == 2:
                                    start, end = int(nums[0]), int(nums[1])
                                    spans.append((start, end))
                                else:
                                    print(f"Weird span format in {filename}: {span}")

                        # Save one row per span
                        for start, end in spans:
                            rows.append({
                                'filename': filename,
                                'annotation_id': annotation_id,
                                'concept_id': concept_id,
                                'term': term,
                                'start_offset': start,
                                'end_offset': end,
                                'text': text_part
                            })
                    else:
                        print(f"Skipping unparseable line in {filename}: {line}")
                except Exception as e:
                    print(f"Error parsing line in {filename}: {line}")
                    print(f"Exception: {e}")
                    continue

# Create DataFrame
df_cadec = pd.DataFrame(rows)

print(f"Parsed {len(df_cadec)} spans.")
#print(df_cadec.head())


Error parsing line in LIPITOR.969.ann: TT1	76948002 | Severe pain |+ 288231001 | Myalgia/myositis - lower leg | 0 30	Severe pain in my calf muscles
Exception: invalid literal for int() with base 10: '+'
Skipping unparseable line in LIPITOR.1000.ann: TT17	CONCEPT_LESS 844 852	crippled
Error parsing line in LIPITOR.766.ann: TT4	274665008 | Chronic intractable pain | + 279039007 | Low back pain | 144 173	irretractable lower back pain
Exception: invalid literal for int() with base 10: '+'
Skipping unparseable line in LIPITOR.766.ann: TT9	CONCEPT_LESS 301 311	detachment
Error parsing line in LIPITOR.772.ann: TT1	76948002 | Severe pain |+ 288226003 | Myalgia/myositis - shoulder 0 34	severe muscle pain in my shoulders
Exception: invalid literal for int() with base 10: '+'
Error parsing line in LIPITOR.564.ann: TT2	76948002 | Severe pain | + 300954003 | Pain in calf | 0 14;20 24	imence pain in calf
Exception: invalid literal for int() with base 10: '+'
Error parsing line in LIPITOR.564.ann: TT

In [21]:
df_cadec.head()

,filename,annotation_id,concept_id,term,start_offset,end_offset,text
0,LIPITOR.95.ann,TT1,449917004,Cramp in lower limb,57,67,leg cramps
1,LIPITOR.95.ann,TT2,55300003,Muscle cramp,158,164,cramps
2,LIPITOR.969.ann,TT2,4031011000036106,Crestor,229,236,Crestor
3,LIPITOR.969.ann,TT3,3904011000036106,Zocor,290,296,Zochor
4,LIPITOR.969.ann,TT4,76948002,Severe pain,332,351,pain was too severe


In [23]:
df_cadec['drug'] = df_cadec['filename'].apply(lambda x: x.split('.')[0].strip().lower())
df_cadec = df_cadec.rename(columns={'term': 'effect'})
df_cadec = df_cadec[['drug','effect']]
df_cadec.head()

,drug,effect
0,lipitor,Cramp in lower limb
1,lipitor,Muscle cramp
2,lipitor,Crestor
3,lipitor,Zocor
4,lipitor,Severe pain


In [25]:
df_cadec.shape

(9450, 2)

### Combinig ADE and CADEC Datasets

In [30]:
#Make sure both 'drug' columns are lowercase and stripped (optional but recommended)
train_df_drug_ade['drug'] = train_df_drug_ade['drug'].str.lower().str.strip()
df_cadec['drug'] = df_cadec['drug'].str.lower().str.strip()

#Concatenate
combined_ADE_CADEC = pd.concat([train_df_drug_ade, df_cadec], ignore_index=True)

In [32]:
print(f"Shape of combined_df: {combined_ADE_CADEC.shape}")

Shape of combined_df: (16271, 2)


In [34]:
combined_ADE_CADEC.head()

,drug,effect
0,azithromycin,ototoxicity
1,dihydrotachysterol,increased calcium-release
2,dihydrotachysterol,hypercalcemia
3,naproxen,pseudoporphyria
4,oxaprozin,pseudoporphyria


In [71]:
#Concatenate
#combined_final= pd.concat([combined_ADE_CADEC, merged_small], ignore_index=True)
combined_final=combined_ADE_CADEC

In [75]:
combined_final.head()

,drug,effect
0,azithromycin,ototoxicity
1,dihydrotachysterol,increased calcium-release
2,dihydrotachysterol,hypercalcemia
3,naproxen,pseudoporphyria
4,oxaprozin,pseudoporphyria


In [77]:
combined_final.tail()

,drug,effect
28618175,acetaminophen\codeine phosphate,toxic epidermal necrolysis
28618176,methylprednisolone,drug reaction with eosinophilia and systemic s...
28618177,methylprednisolone,toxic epidermal necrolysis
28618178,methylprednisolone,drug reaction with eosinophilia and systemic s...
28618179,methylprednisolone,toxic epidermal necrolysis


In [79]:
print(f"Shape of combined_df: {combined_final.shape}")

Shape of combined_df: (28618180, 2)


In [81]:
num_classes = combined_final['effect'].nunique()
print(num_classes)

16701


#### Checking for duplicates and remove

In [84]:
# See how many duplicate drug-effect pairs exist
duplicate_rows = combined_final.duplicated(subset=['drug', 'effect'])

# Print how many
print(f"Number of duplicate drug-effect rows: {duplicate_rows.sum()}")

Number of duplicate drug-effect rows: 27057256


In [86]:
# Remove duplicates based on drug and effect columns
combined_final = combined_final.drop_duplicates(subset=['drug', 'effect'])

# Reset index (optional, for clean numbering)
combined_final = combined_final.reset_index(drop=True)

In [88]:
print(f"Shape of combined_df: {combined_final.shape}")

Shape of combined_df: (1560924, 2)


### Checking for highest use of words

In [91]:
import pandas as pd
from collections import Counter
import re
import nltk
from nltk.corpus import stopwords

# Download stopwords (only the first time you run)
nltk.download('stopwords')

# Define stopwords list
stop_words = set(stopwords.words('english'))

# Assuming your dataframe is called your_dataframe
# 1. Merge all effects into one long string
all_effect_text = " ".join(combined_final['effect'].dropna().tolist())

# 2. Lowercase and remove non-alphabetic characters (keep words only)
all_effect_text = re.sub(r'[^a-z\s]', '', all_effect_text.lower())

# 3. Split into individual words
words = all_effect_text.split()

# 4. Remove stopwords
filtered_words = [word for word in words if word not in stop_words]

# 5. Count word frequencies
word_counts = Counter(filtered_words)

# 6. Display top 100 most common words
print(word_counts.most_common(100))


[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/farhanaalam/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


[('increased', 62113), ('disorder', 60730), ('decreased', 56600), ('infection', 52024), ('blood', 49169), ('product', 45827), ('pain', 45008), ('site', 36516), ('abnormal', 32386), ('drug', 26681), ('syndrome', 25025), ('issue', 23836), ('skin', 23830), ('disease', 20408), ('injury', 19786), ('haemorrhage', 18391), ('device', 17023), ('count', 16392), ('respiratory', 16360), ('infusion', 15874), ('use', 15571), ('fracture', 15088), ('discomfort', 15081), ('swelling', 14395), ('abdominal', 14143), ('tract', 14096), ('cell', 13927), ('injection', 13574), ('pulmonary', 12944), ('failure', 12524), ('reaction', 12394), ('gastrointestinal', 11816), ('pneumonia', 10919), ('pressure', 10917), ('rash', 10594), ('eye', 10493), ('acute', 10366), ('oedema', 10224), ('cardiac', 10123), ('cancer', 9938), ('renal', 9751), ('test', 9646), ('peripheral', 9504), ('hepatic', 9373), ('dose', 9132), ('joint', 8970), ('therapeutic', 8844), ('weight', 8800), ('neoplasm', 8658), ('muscle', 8373), ('positive',

### Preparation for making categories

In [171]:
!pip install spacy

In [420]:
# Count how many 'Other'
other_count = (combined_final['category'] == 'Other').sum()

# Count how many are NOT 'Other'
not_other_count = (combined_final['category'] != 'Other').sum()

print(f"Number of 'Other': {other_count}")
print(f"Number of categorized entries: {not_other_count}")

Number of 'Other': 883449
Number of categorized entries: 677475


### Making Categories based on most frequent words

In [249]:
# with 25
category_keywords = {
    "Infections": ["infection", "pneumonia", "viral", "covid", "bacterial","exposure", "fungal", "parasitic"],
    "Pain Disorders": ["pain", "discomfort", "headache", "arthralgia","migraine"],
    "Skin Disorders": ["rash","cutaneous","skin","hair","erythema","dermnatitis","discolouration","dermatitis","dermatologic","urticaria","eruption","erythema","pustulosis", "pruritus","epidermal","pseudoporphyria"],
    "Hypersensitivity/Allergic Reactions": ["hypersensitivity","angioedema","anaphylaxis","allergy","allergic"],
    "Fracture/Bone Disorders": ["fracture", "bone", "osteoporosis"],
    "Musculoskeletal Disorders": ["muscle", "joint","limb","arthritis", "myopathy","musculoskeletal", "gait"],
    "Respiratory Disorders": ["pulmonary","theophylline", "pneumonitis","lung","dyspnoea","respiratory", "asthma", "bronchitis"],
    "Cardiovascular Disorders": ["cardiac", "ischaemia","ischemia","heart","atrial","vascular", "hypertension","cardiovascular","hypotension", "arrhythmia"],
    "Gastrointestinal Disorders": ["gastrointestinal", "intestinal","enteropathy","endoscopic","ulcer", "diarrhoea", "nausea", "vomiting","abdominal", "hepatic", "tract"],
    "Liver Disorders": ["liver", "hepatic", "hepatitis"],
    "Renal Disorders": ["renal", "kidney", "nephropathy"],
    "Blood Disorders": ["blood","thrombocytopenia","thrombotic","hemolytic","leukopenia","anemia","bicytopenia", "hemorrhagic", "bleeding", "hemorrhage","neutropenia", "platelet","haematocrit","hypercalcemia","granulocytopenia","haemoglobin","methemoglobinemia","thalassaemia"],
    "Cancer Related Events": ["cancer", "tumor", "neoplasm", "leukemia", "lymphoma"],
    "Neurological Disorders": ["neurotoxicity","spinal","seizure","nerve","brain","ataxia","encephalopathy","impaired","neurologic", "neuropathy", "dizziness", "sleep"],
    "Psychiatric Disorders": ["depression","psychotic","psychiatric","anxiety","delirious","psychosis","encephalopathy","insomnia"],
    "Fatigue/Weakness": ["fatigue", "malaise", "tiredness"],
    "Eye Disorders": ["eye","glaucoma","visual","vision","ectropion","retinopathy"],
    "Thrombosis/Clotting Disorders": ["thrombosis", "embolism", "deep vein", "clot"],
    "Immune System Disorders": ["immune","scleroderma","antibodies", "immunodeficiency", "autoimmune"],
    "Reproductive Disorders": ["pregnancy","polycystic","testosterone","reproductive","congenital","menstrual","ovary","miscarriage", "fertility"],
    "Device/Procedure Related Issues": ["device", "implant", "catheter", "prosthesis"],
    "Therapeutic Complications": ["therapy", "treatment", "drug reaction", "adverse"],
    "Fever/Temperature Related": ["fever", "temperature", "pyrexia"],
    "Metabolic Disorders": ["diabetes","hypocalcemic","hypophosphatemia","diarrhea","acidosis","pancreatitis","hyperglycemia","creatine","hypoglycemia"],
    "Other": []  # fallback for anything unmatched
}

In [251]:
def categorize_effect(effect):
    effect = str(effect).lower()
    for category, keywords in category_keywords.items():
        for keyword in keywords:
            if keyword in effect:
                return category
    return "Other"

combined_final['category'] = combined_final['effect'].apply(categorize_effect)

In [148]:
combined_final.tail()

,drug,effect,category
1560919,ampicillin,haemoglobin decreased,Blood Disorders
1560920,ampicillin,haematocrit decreased,Blood Disorders
1560921,ampicillin,transferrin decreased,Other
1560922,ampicillin,thalassaemia minor,Blood Disorders
1560923,ampicillin,normal newborn,Other


In [257]:
# Count how many 'Other'
other_count = (combined_final['category'] == 'Other').sum()

# Count how many are NOT 'Other'
not_other_count = (combined_final['category'] != 'Other').sum()

print(f"Number of 'Other': {other_count}")
print(f"Number of categorized entries: {not_other_count}")

Number of 'Other': 915373
Number of categorized entries: 645551


### Saving combined dataset as a csv file

In [ ]:
combined_final.to_csv('combined_final.csv', index=False)

In [ ]:
combined_final.to_csv('combined_final_woDuplicattes.csv', index=False)